# BigQuery Catalog Mapper — Claude-Code Harness over a Financial Data Warehouse

A long-running data-modeling agent built on the same single-notebook Claude-Code
harness as `claude_code_from_scratch.ipynb`. It points at a **BigQuery warehouse
with hundreds of tables** and works through it to:

1. **Make sense of the tables** — infer each table's purpose and grain from its
   schema, descriptions, and a small data sample.
2. **Surface business rules** — encode constraints/semantics it observes (status
   enums, nullability, monetary units, date grains, soft-deletes, etc.).
3. **Find the principal join attributes** — discover candidate keys and the
   columns that link tables together, ranked by confidence.

It uses the base harness's planning / long-horizon machinery: `todo_write` for the
current pass, the durable `task_*` graph for multi-session coverage across
hundreds of tables, rolling context compaction, and `spawn_subagent` to
deep-profile one table/dataset without polluting the lead's context.

### Cost posture — metadata + capped sampling
BigQuery bills by **bytes scanned**, so the tools are built defensively:
- **Free metadata** does the heavy lifting: schemas, row counts, descriptions,
  partitioning/clustering, and `INFORMATION_SCHEMA` column inventory (no data read).
- Any tool that reads **data** (sample / profile / ad-hoc query / value-overlap
  join check) is **dry-run first** and **refused if it would scan more than
  `BQ_MAX_BYTES_BILLED`** (default 1 GiB); the job also sets `maximum_bytes_billed`
  as a hard stop. Only read-only `SELECT`/`WITH` is permitted.

### Tools added on top of the base harness

| Group | Tool | Cost |
|---|---|---|
| **Inspect (free)** | `bq_list_datasets`, `bq_list_tables`, `bq_table_schema`, `bq_columns`, `bq_estimate` | metadata only |
| **Read data (capped)** | `bq_sample`, `bq_query`, `bq_join_overlap` | dry-run + `maximum_bytes_billed` |
| **Build the catalog** | `catalog_set_table`, `catalog_add_join`, `catalog_summary`, `catalog_export` | local |

### Outputs (written to the sandbox)
`data_catalog.json` (always) · `data_dictionary.md` · `join_candidates.csv` ·
`suggested_views.sql`.

> **Setup:** `pip install google-cloud-bigquery`, then edit the config cell:
> `BQ_PROJECT`, optional `BQ_DATASETS`, `BQ_LOCATION`, and auth (ADC via
> `gcloud auth application-default login`, or a service-account JSON in
> `GOOGLE_APPLICATION_CREDENTIALS`). Read-only access is sufficient.


In [ ]:
"""
Imports + logging. One root logger "agent"; child loggers per subsystem.
Set AGENT_LOG_LEVEL=DEBUG for full payloads.
"""
from __future__ import annotations

import json
import logging
import os
import re
import subprocess
import sys
import time
import uuid
import glob as _glob
from collections import defaultdict
from pathlib import Path
from typing import Any, Callable, Dict, List, Optional, Tuple

import requests  # used only to talk to Ollama (chat). BigQuery uses its client.

try:
    from dotenv import load_dotenv, find_dotenv
    _dotenv = find_dotenv(usecwd=True)
    if _dotenv:
        load_dotenv(_dotenv)
except Exception:
    pass


AGENT_LOG_LEVEL = os.environ.get("AGENT_LOG_LEVEL", "INFO").upper()

class _Fmt(logging.Formatter):
    COLORS = {
        "DEBUG":    "\033[90m", "INFO": "\033[36m", "WARNING": "\033[33m",
        "ERROR":    "\033[31m", "CRITICAL": "\033[41m",
    }
    RESET = "\033[0m"
    def format(self, record: logging.LogRecord) -> str:
        color = self.COLORS.get(record.levelname, "")
        short = record.name.split(".", 1)[1] if "." in record.name else record.name
        return f"{color}[{record.levelname:5s}] {short:18s} | {record.getMessage()}{self.RESET}"

_handler = logging.StreamHandler(sys.stdout)
_handler.setFormatter(_Fmt())

log = logging.getLogger("agent")
log.handlers.clear()
log.addHandler(_handler)
log.setLevel(AGENT_LOG_LEVEL)
log.propagate = False

log_loop    = logging.getLogger("agent.loop")
log_ollama  = logging.getLogger("agent.ollama")
log_tool    = logging.getLogger("agent.tool")
log_bq      = logging.getLogger("agent.bq")
log_cat     = logging.getLogger("agent.catalog")
log_sub     = logging.getLogger("agent.subagent")
log_todo    = logging.getLogger("agent.todo")
log_compact = logging.getLogger("agent.compact")
log_chat    = logging.getLogger("agent.chat")

log.info(f"Logger initialised at level {AGENT_LOG_LEVEL}")


In [ ]:
"""
Configuration. All knobs live here.
"""

# --- Ollama ------------------------------------------------------------------
OLLAMA_HOST = os.environ.get("OLLAMA_HOST", "http://localhost:8080")
MODELS = {
    "lead":       os.environ.get("AGENT_LEAD_MODEL",       "qwen3:32b"),
    "subagent":   os.environ.get("AGENT_SUBAGENT_MODEL",   "qwen3:8b"),
    "summarizer": os.environ.get("AGENT_SUMMARIZER_MODEL", "qwen3:8b"),
}

# --- Sandbox / outputs -------------------------------------------------------
WORKDIR = Path(os.environ.get("AGENT_WORKDIR", str(Path.cwd() / "sandbox"))).resolve()
WORKDIR.mkdir(parents=True, exist_ok=True)
SKILLS_DIR = Path(os.environ.get("AGENT_SKILLS_DIR", str(WORKDIR / "skills")))

# --- State files (persisted across runs) ------------------------------------
TODO_FILE    = WORKDIR / ".agent_todo.json"
TASKS_FILE   = WORKDIR / ".agent_tasks.json"
MEMORY_FILE  = WORKDIR / ".agent_memory.md"
CATALOG_FILE = WORKDIR / "data_catalog.json"   # the live, resumable catalog

# --- Limits ------------------------------------------------------------------
MAX_TOOL_OUTPUT     = 20_000
MAX_TURNS           = 40       # hundreds of tables -> long-horizon
COMPRESS_THRESHOLD  = 45_000
KEEP_RECENT         = 6
BASH_TIMEOUT_S      = 60
REQUEST_TIMEOUT_S   = 600

# --- BigQuery ----------------------------------------------------------------
# Auth: ADC (`gcloud auth application-default login`) or a service-account JSON
# pointed to by GOOGLE_APPLICATION_CREDENTIALS. Read-only access is enough.
BQ_PROJECT     = os.environ.get("BQ_PROJECT") or os.environ.get("GOOGLE_CLOUD_PROJECT")
BQ_LOCATION    = os.environ.get("BQ_LOCATION", "US")
# Comma-separated allow-list of datasets to scope the work; empty = all in project.
BQ_DATASETS    = [d.strip() for d in os.environ.get("BQ_DATASETS", "").split(",") if d.strip()]
BQ_CREDENTIALS_PATH = os.environ.get("GOOGLE_APPLICATION_CREDENTIALS")  # optional explicit SA json

# Cost controls (posture: metadata + capped sampling).
BQ_MAX_BYTES_BILLED = int(os.environ.get("BQ_MAX_BYTES_BILLED", str(1 << 30)))  # 1 GiB hard cap
BQ_SAMPLE_ROWS      = int(os.environ.get("BQ_SAMPLE_ROWS", "8"))
BQ_QUERY_ROW_LIMIT  = int(os.environ.get("BQ_QUERY_ROW_LIMIT", "200"))
BQ_MAX_DATASETS_SCAN = int(os.environ.get("BQ_MAX_DATASETS_SCAN", "60"))  # for cross-dataset column search

BASH_BLOCKLIST = ["rm -rf /", "sudo", "shutdown", "reboot", "mkfs", "> /dev/", ":(){ :|:& };:"]

log.info(f"OLLAMA_HOST = {OLLAMA_HOST}")
log.info(f"WORKDIR     = {WORKDIR}")
log.info(f"MODELS      = {MODELS}")
log.info(f"BQ_PROJECT  = {BQ_PROJECT!r}  LOCATION={BQ_LOCATION}  DATASETS={BQ_DATASETS or '(all)'}")
log.info(f"BQ cost cap = {BQ_MAX_BYTES_BILLED} bytes ({BQ_MAX_BYTES_BILLED/(1<<30):.2f} GiB) per data query")


In [ ]:
"""
Ollama client (chat). Thin wrapper over /api/chat with OpenAI-style tool calling.
"""

def ollama_chat(*, model: str, messages: List[Dict[str, Any]],
                tools: Optional[List[Dict[str, Any]]] = None,
                temperature: float = 0.2) -> Dict[str, Any]:
    payload: Dict[str, Any] = {
        "model": model, "messages": messages, "stream": False,
        "options": {"temperature": temperature},
    }
    if tools:
        payload["tools"] = tools
    approx = sum(len(str(m.get("content", ""))) for m in messages)
    log_ollama.info(f"-> {model}  msgs={len(messages)}  ~{approx} chars  "
                    f"tools={len(tools) if tools else 0}")
    t0 = time.time()
    resp = requests.post(f"{OLLAMA_HOST}/api/chat", json=payload, timeout=REQUEST_TIMEOUT_S)
    dt = time.time() - t0
    if resp.status_code != 200:
        log_ollama.error(f"<- HTTP {resp.status_code}: {resp.text[:500]}")
        resp.raise_for_status()
    data = resp.json()
    msg = data.get("message", {})
    tcs = msg.get("tool_calls") or []
    log_ollama.info(f"<- {model}  {dt:5.1f}s  text={len(msg.get('content') or '')}ch  "
                    f"tool_calls={len(tcs)}  done_reason={data.get('done_reason','?')}")
    return msg


def ollama_healthcheck() -> bool:
    try:
        r = requests.get(f"{OLLAMA_HOST}/api/tags", timeout=5)
        r.raise_for_status()
        tags = [m["name"] for m in r.json().get("models", [])]
        log_ollama.info(f"healthcheck OK -- {len(tags)} models available")
        for role, name in MODELS.items():
            present = any(t == name or t.startswith(name + ":") for t in tags)
            log_ollama.info(f"   {role:11s} {name:18s} [{'OK' if present else 'MISSING'}]")
        return True
    except Exception as e:
        log_ollama.error(f"healthcheck FAILED: {e}")
        return False

ollama_healthcheck()


In [ ]:
"""
BigQuery access layer (google-cloud-bigquery) with cost guards.

Free (metadata-only) tools:
  bq_list_datasets()                          datasets in the project/allow-list
  bq_list_tables(dataset, max_tables)         table names + types in a dataset
  bq_table_schema(table)                      columns/types/desc, row count, size,
                                              partitioning/clustering, labels
  bq_columns(name_like, dataset)              INFORMATION_SCHEMA column inventory
                                              search (join-key discovery)
  bq_estimate(sql)                            dry-run byte estimate (no data read)

Data-reading tools (DRY-RUN FIRST, refused above BQ_MAX_BYTES_BILLED, and run with
maximum_bytes_billed set; read-only SELECT/WITH only):
  bq_sample(table, n)                         a few example rows
  bq_query(sql, row_limit)                    ad-hoc read-only query
  bq_join_overlap(lt, lcol, rt, rcol, sample) value-overlap FK check on a sample

`_truncate` / `MAX_TOOL_OUTPUT` come from the core-tools cell.
"""

try:
    from google.cloud import bigquery
    _BQ_AVAILABLE = True
    _BQ_IMPORT_ERR = None
except Exception as _e:          # ImportError or transitive dep issue
    bigquery = None
    _BQ_AVAILABLE = False
    _BQ_IMPORT_ERR = _e

_bq_client_obj = None


def bq_client():
    if not _BQ_AVAILABLE:
        raise RuntimeError(
            "google-cloud-bigquery is not installed. Run: pip install google-cloud-bigquery"
            f"  (import error: {_BQ_IMPORT_ERR})")
    global _bq_client_obj
    if _bq_client_obj is None:
        kwargs: Dict[str, Any] = {}
        if BQ_LOCATION:
            kwargs["location"] = BQ_LOCATION
        if BQ_CREDENTIALS_PATH:
            from google.oauth2 import service_account
            creds = service_account.Credentials.from_service_account_file(BQ_CREDENTIALS_PATH)
            kwargs["credentials"] = creds
            kwargs["project"] = BQ_PROJECT or creds.project_id
        elif BQ_PROJECT:
            kwargs["project"] = BQ_PROJECT
        _bq_client_obj = bigquery.Client(**kwargs)
        log_bq.info(f"BigQuery client ready (project={_bq_client_obj.project}, location={BQ_LOCATION})")
    return _bq_client_obj


def _h(n: Optional[float]) -> str:
    n = float(n or 0)
    for u in ("B", "KB", "MB", "GB", "TB", "PB"):
        if n < 1024:
            return f"{n:.1f} {u}"
        n /= 1024
    return f"{n:.1f} EB"


_IDENT_RE = re.compile(r"^[A-Za-z_][A-Za-z0-9_]*$")
_LIKE_RE  = re.compile(r"^[A-Za-z0-9_%]+$")
_READONLY_RE  = re.compile(r"^\s*(with|select)\b", re.IGNORECASE)
_FORBIDDEN_RE = re.compile(r"\b(insert|update|delete|merge|drop|create|alter|truncate|grant|revoke|call|load|export|begin|declare)\b",
                           re.IGNORECASE)


def _safe_ident(s: str) -> str:
    if not _IDENT_RE.match(s or ""):
        raise ValueError(f"unsafe identifier: {s!r}")
    return s


def _resolve_table(table: str) -> Tuple[str, str, str]:
    parts = table.replace("`", "").strip().split(".")
    if len(parts) == 3:
        proj, ds, tbl = parts
    elif len(parts) == 2:
        proj = BQ_PROJECT or bq_client().project
        ds, tbl = parts
    else:
        raise ValueError(f"table must be 'dataset.table' or 'project.dataset.table', got {table!r}")
    return proj, _safe_ident(ds), _safe_ident(tbl)


def _fqn(proj: str, ds: str, tbl: str) -> str:
    return f"`{proj}`.`{ds}`.`{tbl}`"


def _dry_run_bytes(sql: str) -> int:
    cfg = bigquery.QueryJobConfig(dry_run=True, use_query_cache=False)
    job = bq_client().query(sql, job_config=cfg)
    return int(job.total_bytes_processed or 0)


def _guarded_query(sql: str, row_limit: int = BQ_QUERY_ROW_LIMIT,
                   cap: Optional[int] = None) -> Tuple[Optional[list], str]:
    """Returns (rows, info). rows is None on refusal/error; info always set."""
    if not _READONLY_RE.match(sql or "") or _FORBIDDEN_RE.search(sql or ""):
        return None, "Error: only read-only SELECT/WITH queries are allowed."
    cap = cap or BQ_MAX_BYTES_BILLED
    try:
        nbytes = _dry_run_bytes(sql)
    except Exception as e:
        return None, f"Error: dry-run failed: {e}"
    if nbytes > cap:
        return None, (f"REFUSED: would scan ~{_h(nbytes)} (> cap {_h(cap)}). Narrow it: select "
                      "fewer columns, add a WHERE/partition filter, use TABLESAMPLE or a smaller LIMIT.")
    cfg = bigquery.QueryJobConfig(maximum_bytes_billed=cap, use_query_cache=True)
    try:
        job = bq_client().query(sql, job_config=cfg)
        rows = list(job.result(max_results=row_limit))
    except Exception as e:
        return None, f"Error: query failed: {e}"
    billed = getattr(job, "total_bytes_billed", None)
    return rows, f"(scanned ~{_h(nbytes)}, billed ~{_h(billed)}, {len(rows)} row(s))"


def _rows_to_table(rows: list, max_rows: int = 50) -> str:
    if not rows:
        return "(no rows)"
    cols = list(rows[0].keys())
    out = [" | ".join(cols), " | ".join("---" for _ in cols)]
    for r in rows[:max_rows]:
        out.append(" | ".join(str(r.get(c))[:60] for c in cols))
    if len(rows) > max_rows:
        out.append(f"... (+{len(rows) - max_rows} more rows)")
    return "\n".join(out)


# ---- free / metadata tools --------------------------------------------------
def tool_bq_list_datasets() -> str:
    log_bq.info("[list_datasets]")
    try:
        client = bq_client()
        names = [d.dataset_id for d in client.list_datasets()]
    except Exception as e:
        return f"Error: {e}"
    if BQ_DATASETS:
        allow = set(BQ_DATASETS)
        names = [n for n in names if n in allow]
    return (f"project={bq_client().project}  datasets ({len(names)}):\n" +
            "\n".join(f"  {n}" for n in sorted(names))) if names else "(no datasets visible)"


def tool_bq_list_tables(dataset: str, max_tables: int = 300) -> str:
    log_bq.info(f"[list_tables] {dataset}")
    try:
        ds = _safe_ident(dataset)
        client = bq_client()
        items = list(client.list_tables(f"{client.project}.{ds}"))
    except Exception as e:
        return f"Error: {e}"
    rows = [f"{it.table_id}\t[{it.table_type}]" for it in items[:max_tables]]
    extra = "" if len(items) <= max_tables else f"\n... (+{len(items)-max_tables} more)"
    return (f"{ds}: {len(items)} table(s)\n" + "\n".join(f"  {r}" for r in rows) + extra) if items \
        else f"{ds}: (no tables)"


def tool_bq_table_schema(table: str) -> str:
    log_bq.info(f"[schema] {table}")
    try:
        proj, ds, tbl = _resolve_table(table)
        t = bq_client().get_table(f"{proj}.{ds}.{tbl}")   # free metadata call
    except Exception as e:
        return f"Error: {e}"
    out = [f"{proj}.{ds}.{tbl}  [{t.table_type}]",
           f"rows={t.num_rows:,}  size={_h(t.num_bytes)}"]
    if t.description:
        out.append(f"description: {t.description}")
    if t.labels:
        out.append(f"labels: {dict(t.labels)}")
    if t.time_partitioning:
        out.append(f"partitioned by: {t.time_partitioning.field or '_PARTITIONTIME'} "
                   f"({t.time_partitioning.type_})")
    if t.range_partitioning:
        out.append(f"range-partitioned by: {t.range_partitioning.field}")
    if t.clustering_fields:
        out.append(f"clustered by: {t.clustering_fields}")
    out.append("columns:")

    def _emit(fields, prefix=""):
        for f in fields:
            desc = f" -- {f.description}" if f.description else ""
            out.append(f"  {prefix}{f.name}  {f.field_type}  {f.mode}{desc}")
            if f.field_type == "RECORD" and f.fields:
                _emit(f.fields, prefix + f.name + ".")
    _emit(t.schema)
    return _truncate("\n".join(out))


def tool_bq_columns(name_like: str, dataset: Optional[str] = None) -> str:
    """Find columns across datasets whose name matches a LIKE pattern (join-key discovery)."""
    log_bq.info(f"[columns] like={name_like!r} dataset={dataset!r}")
    if not _LIKE_RE.match(name_like or ""):
        return "Error: name_like may contain only letters, digits, underscore and % (LIKE pattern)."
    try:
        client = bq_client()
        if dataset and dataset not in ("", "*"):
            datasets = [_safe_ident(dataset)]
        else:
            datasets = BQ_DATASETS or [d.dataset_id for d in client.list_datasets()]
        datasets = datasets[:BQ_MAX_DATASETS_SCAN]
    except Exception as e:
        return f"Error: {e}"
    hits: List[str] = []
    scanned = 0
    for ds in datasets:
        sql = (f"SELECT table_name, column_name, data_type "
               f"FROM `{client.project}`.`{ds}`.INFORMATION_SCHEMA.COLUMNS "
               f"WHERE column_name LIKE '{name_like}' "
               f"ORDER BY column_name, table_name")
        rows, info = _guarded_query(sql, row_limit=1000)
        scanned += 1
        if rows is None:
            continue   # dataset may not exist / no access; skip
        for r in rows:
            hits.append(f"  {ds}.{r['table_name']}.{r['column_name']}  ({r['data_type']})")
    if not hits:
        return f"No columns matching {name_like!r} found across {scanned} dataset(s)."
    # group by column name for join visibility
    return (f"{len(hits)} column(s) matching {name_like!r} across {scanned} dataset(s):\n"
            + "\n".join(hits[:400]) + ("" if len(hits) <= 400 else f"\n... (+{len(hits)-400} more)"))


def tool_bq_estimate(sql: str) -> str:
    log_bq.info("[estimate]")
    if not _READONLY_RE.match(sql or "") or _FORBIDDEN_RE.search(sql or ""):
        return "Error: only read-only SELECT/WITH queries can be estimated."
    try:
        nbytes = _dry_run_bytes(sql)
    except Exception as e:
        return f"Error: dry-run failed: {e}"
    verdict = "within cap" if nbytes <= BQ_MAX_BYTES_BILLED else "OVER CAP -- would be refused"
    return f"Estimated scan: {_h(nbytes)} (cap {_h(BQ_MAX_BYTES_BILLED)}) -> {verdict}."


# ---- capped data-reading tools ---------------------------------------------
def tool_bq_sample(table: str, n: int = BQ_SAMPLE_ROWS) -> str:
    log_bq.info(f"[sample] {table} n={n}")
    try:
        proj, ds, tbl = _resolve_table(table)
    except Exception as e:
        return f"Error: {e}"
    n = max(1, min(int(n), 50))
    sql = f"SELECT * FROM {_fqn(proj, ds, tbl)} LIMIT {n}"
    rows, info = _guarded_query(sql, row_limit=n)
    if rows is None:
        return info
    return f"sample of {proj}.{ds}.{tbl} {info}:\n" + _truncate(_rows_to_table(rows, n))


def tool_bq_query(sql: str, row_limit: int = BQ_QUERY_ROW_LIMIT) -> str:
    log_bq.info(f"[query] {sql[:120]}")
    rows, info = _guarded_query(sql, row_limit=row_limit)
    if rows is None:
        return info
    return f"{info}:\n" + _truncate(_rows_to_table(rows, row_limit))


def tool_bq_join_overlap(left_table: str, left_col: str, right_table: str, right_col: str,
                         sample: int = 100_000) -> str:
    """Estimate value overlap between two columns on a sample -> evidence for a FK/join."""
    log_bq.info(f"[overlap] {left_table}.{left_col} ~ {right_table}.{right_col}")
    try:
        lp, lds, lt = _resolve_table(left_table)
        rp, rds, rt = _resolve_table(right_table)
        lc, rc = _safe_ident(left_col), _safe_ident(right_col)
    except Exception as e:
        return f"Error: {e}"
    sample = max(1000, min(int(sample), 2_000_000))
    sql = (
        f"WITH l AS (SELECT DISTINCT CAST(`{lc}` AS STRING) v FROM {_fqn(lp, lds, lt)} "
        f"WHERE `{lc}` IS NOT NULL LIMIT {sample}), "
        f"r AS (SELECT DISTINCT CAST(`{rc}` AS STRING) v FROM {_fqn(rp, rds, rt)} "
        f"WHERE `{rc}` IS NOT NULL LIMIT {sample}) "
        f"SELECT (SELECT COUNT(*) FROM l) AS l_distinct, "
        f"(SELECT COUNT(*) FROM r) AS r_distinct, "
        f"(SELECT COUNT(*) FROM l JOIN r USING (v)) AS overlap"
    )
    rows, info = _guarded_query(sql, row_limit=1)
    if rows is None:
        return info
    r = rows[0]
    ld, rd, ov = r["l_distinct"], r["r_distinct"], r["overlap"]
    lpct = (100.0 * ov / ld) if ld else 0.0
    rpct = (100.0 * ov / rd) if rd else 0.0
    return (f"overlap {info}: left_distinct={ld:,} right_distinct={rd:,} overlap={ov:,}\n"
            f"  {lpct:.1f}% of left values found in right; {rpct:.1f}% of right found in left.\n"
            f"  (high left->right coverage suggests {left_table}.{left_col} is a FK into "
            f"{right_table}.{right_col}.)")

log.info("BigQuery tools defined (metadata + capped sampling).")


In [ ]:
"""
Catalog store + tools.

The catalog is a JSON document in the sandbox (CATALOG_FILE) -- durable, so a
multi-day mapping run can be paused/resumed. Shape:

  {
    "tables": { "project.dataset.table": {
        "purpose": str, "grain": str, "row_count": int,
        "key_columns": [..], "business_rules": [..], "notes": str } },
    "joins":  [ {"left","left_col","right","right_col","confidence","basis","evidence"} ]
  }

confidence : high | medium | low
basis      : name+type | fk-naming | pk-unique | value-overlap | manual

Tools:
  catalog_set_table(table, purpose, grain, row_count, key_columns, business_rules, notes)
  catalog_add_join(left, left_col, right, right_col, confidence, basis, evidence)
  catalog_summary()
  catalog_export(format)   format = json | dictionary | joins | sql | all
"""

JOIN_CONFIDENCE = ("high", "medium", "low")
JOIN_BASIS = ("name+type", "fk-naming", "pk-unique", "value-overlap", "manual")


def _empty_catalog() -> Dict[str, Any]:
    return {"tables": {}, "joins": []}


def _load_catalog() -> Dict[str, Any]:
    if not CATALOG_FILE.exists():
        return _empty_catalog()
    try:
        c = json.loads(CATALOG_FILE.read_text(encoding="utf-8"))
        c.setdefault("tables", {}); c.setdefault("joins", [])
        return c
    except (json.JSONDecodeError, OSError) as e:
        log_cat.warning(f"catalog unreadable, starting empty: {e}")
        return _empty_catalog()


def _save_catalog(c: Dict[str, Any]) -> None:
    CATALOG_FILE.write_text(json.dumps(c, indent=2), encoding="utf-8")


def _as_list(v) -> List[str]:
    if v is None:
        return []
    if isinstance(v, str):
        return [v]
    return list(v)


def run_catalog_set_table(table: str, purpose: Optional[str] = None, grain: Optional[str] = None,
                          row_count: Optional[int] = None, key_columns=None,
                          business_rules=None, notes: Optional[str] = None) -> str:
    c = _load_catalog()
    t = c["tables"].get(table, {"purpose": "", "grain": "", "row_count": None,
                                "key_columns": [], "business_rules": [], "notes": ""})
    if purpose:
        t["purpose"] = purpose
    if grain:
        t["grain"] = grain
    if row_count is not None:
        t["row_count"] = row_count
    if key_columns:
        for k in _as_list(key_columns):
            if k not in t["key_columns"]:
                t["key_columns"].append(k)
    if business_rules:
        for br in _as_list(business_rules):
            if br not in t["business_rules"]:
                t["business_rules"].append(br)
    if notes:
        t["notes"] = (t.get("notes", "") + ("\n" if t.get("notes") else "") + notes)
    c["tables"][table] = t
    _save_catalog(c)
    log_cat.info(f"[table] {table}")
    return (f"catalogued {table}: purpose={'set' if t['purpose'] else '-'} grain={'set' if t['grain'] else '-'} "
            f"keys={t['key_columns']} rules={len(t['business_rules'])}")


def run_catalog_add_join(left: str, left_col: str, right: str, right_col: str,
                         confidence: str = "medium", basis: str = "name+type",
                         evidence: str = "") -> str:
    if confidence not in JOIN_CONFIDENCE:
        return f"Error: confidence must be one of {JOIN_CONFIDENCE}"
    if basis not in JOIN_BASIS:
        return f"Error: basis must be one of {JOIN_BASIS}"
    c = _load_catalog()
    for j in c["joins"]:
        if (j["left"], j["left_col"], j["right"], j["right_col"]) == (left, left_col, right, right_col):
            j["confidence"] = confidence
            j["basis"] = basis
            if evidence:
                j["evidence"] = evidence
            _save_catalog(c)
            return f"join updated: {left}.{left_col} = {right}.{right_col} [{confidence}/{basis}]"
    c["joins"].append({"left": left, "left_col": left_col, "right": right, "right_col": right_col,
                       "confidence": confidence, "basis": basis, "evidence": evidence})
    _save_catalog(c)
    log_cat.info(f"[join] {left}.{left_col} = {right}.{right_col} [{confidence}/{basis}]")
    return f"join added: {left}.{left_col} = {right}.{right_col} [{confidence}/{basis}]"


def run_catalog_summary() -> str:
    c = _load_catalog()
    tables, joins = c["tables"], c["joins"]
    described = sum(1 for t in tables.values() if t.get("purpose"))
    ruled = sum(len(t.get("business_rules", [])) for t in tables.values())
    by_conf: Dict[str, int] = {}
    for j in joins:
        by_conf[j["confidence"]] = by_conf.get(j["confidence"], 0) + 1
    deg: Dict[str, int] = defaultdict(int)
    for j in joins:
        deg[j["left"]] += 1
        deg[j["right"]] += 1
    hubs = sorted(deg.items(), key=lambda kv: -kv[1])[:5]
    return ("\n".join([
        f"tables catalogued: {len(tables)} ({described} with a purpose)",
        f"business rules captured: {ruled}",
        f"join candidates: {len(joins)} by confidence {dict(by_conf)}",
        f"most-connected tables (hubs): {hubs}",
    ]))


def _md(s: Any) -> str:
    return str(s).replace("|", "/").replace("\n", " ")


def _to_dictionary(c: Dict[str, Any]) -> str:
    tables = c["tables"]
    L = ["# Data Dictionary", "", f"_{len(tables)} tables catalogued._", ""]
    for name in sorted(tables):
        t = tables[name]
        L.append(f"## `{name}`")
        if t.get("row_count") is not None:
            L.append(f"- **Rows:** {t['row_count']:,}")
        if t.get("purpose"):
            L.append(f"- **Purpose:** {t['purpose']}")
        if t.get("grain"):
            L.append(f"- **Grain:** {t['grain']}")
        if t.get("key_columns"):
            L.append(f"- **Key columns:** {', '.join('`'+k+'`' for k in t['key_columns'])}")
        if t.get("business_rules"):
            L.append("- **Business rules:**")
            L += [f"    - {br}" for br in t["business_rules"]]
        if t.get("notes"):
            L.append(f"- **Notes:** {t['notes']}")
        L.append("")
    return "\n".join(L)


def _to_join_csv(c: Dict[str, Any]) -> str:
    order = {"high": 0, "medium": 1, "low": 2}
    rows = sorted(c["joins"], key=lambda j: (order.get(j["confidence"], 9), j["left"], j["left_col"]))
    out = ["left_table,left_col,right_table,right_col,confidence,basis,evidence"]
    for j in rows:
        ev = str(j.get("evidence", "")).replace('"', "'").replace("\n", " ")
        out.append(f'"{j["left"]}","{j["left_col"]}","{j["right"]}","{j["right_col"]}",'
                   f'{j["confidence"]},{j["basis"]},"{ev}"')
    return "\n".join(out)


def _to_view_sql(c: Dict[str, Any]) -> str:
    joins = c["joins"]
    lines = ["-- Suggested JOINs / starter views derived from discovered keys.",
             "-- Review before use; joins are inferred, not enforced constraints.", ""]
    # 1) one snippet per recorded join
    lines.append("-- ---- pairwise join snippets ----")
    for j in joins:
        lines.append(f"-- [{j['confidence']}/{j['basis']}] {j.get('evidence','')}".rstrip())
        lines.append(f"SELECT l.*, r.*")
        lines.append(f"FROM `{j['left']}` AS l")
        lines.append(f"JOIN `{j['right']}` AS r")
        lines.append(f"  ON l.`{j['left_col']}` = r.`{j['right_col']}`;")
        lines.append("")
    # 2) a hub view: star-join the most-connected table to its neighbours
    deg: Dict[str, int] = defaultdict(int)
    for j in joins:
        deg[j["left"]] += 1
        deg[j["right"]] += 1
    if deg:
        hub = max(deg.items(), key=lambda kv: kv[1])[0]
        neigh = [j for j in joins if j["left"] == hub or j["right"] == hub]
        if neigh:
            lines.append(f"-- ---- hub view around the most-connected table: {hub} ----")
            lines.append("CREATE OR REPLACE VIEW `your_dataset.vw_hub` AS")
            lines.append("SELECT h.*")
            lines.append(f"FROM `{hub}` AS h")
            for i, j in enumerate(neigh):
                other = j["right"] if j["left"] == hub else j["left"]
                hub_col = j["left_col"] if j["left"] == hub else j["right_col"]
                oth_col = j["right_col"] if j["left"] == hub else j["left_col"]
                lines.append(f"LEFT JOIN `{other}` AS t{i} ON h.`{hub_col}` = t{i}.`{oth_col}`")
            lines[-1] = lines[-1] + ";"
            lines.append("")
    return "\n".join(lines)


def run_catalog_export(format: str = "all") -> str:
    c = _load_catalog()
    fmt = (format or "all").lower()
    written = []
    if fmt in ("json", "all"):
        p = WORKDIR / "data_catalog.json"
        p.write_text(json.dumps(c, indent=2), encoding="utf-8")
        written.append(str(p))
    if fmt in ("dictionary", "md", "markdown", "all"):
        p = WORKDIR / "data_dictionary.md"
        p.write_text(_to_dictionary(c), encoding="utf-8")
        written.append(str(p))
    if fmt in ("joins", "csv", "all"):
        p = WORKDIR / "join_candidates.csv"
        p.write_text(_to_join_csv(c), encoding="utf-8")
        written.append(str(p))
    if fmt in ("sql", "views", "all"):
        p = WORKDIR / "suggested_views.sql"
        p.write_text(_to_view_sql(c), encoding="utf-8")
        written.append(str(p))
    if not written:
        return f"Error: unknown format {format!r}. Use json | dictionary | joins | sql | all."
    log_cat.info(f"[export] {fmt} -> {written}")
    return "Exported:\n" + "\n".join(f"  {w}" for w in written) + "\n\n--- summary ---\n" + run_catalog_summary()

log.info("Catalog tools defined: catalog_set_table, catalog_add_join, catalog_summary, catalog_export")


In [ ]:
"""
Core file & shell tools (sandboxed to WORKDIR). Let the agent persist working
notes/reports and post-process the exported CSV/SQL.
"""

SNAPSHOTS: Dict[str, Optional[str]] = {}


def _safe_path(path: str) -> Path:
    p = (WORKDIR / path).resolve() if not os.path.isabs(path) else Path(path).resolve()
    try:
        p.relative_to(WORKDIR)
    except ValueError:
        raise ValueError(f"path escapes WORKDIR: {p}")
    return p


def _truncate(s: str, limit: int = MAX_TOOL_OUTPUT) -> str:
    if len(s) <= limit:
        return s
    return s[:limit] + f"\n... [truncated {len(s) - limit} chars]"


def tool_bash(command: str) -> str:
    log_tool.info(f"[bash] {command[:120]}")
    for bad in BASH_BLOCKLIST:
        if bad in command:
            return f"Error: blocked by safety policy (matched {bad!r})"
    try:
        proc = subprocess.run(command, shell=True, cwd=str(WORKDIR),
                              capture_output=True, text=True, timeout=BASH_TIMEOUT_S)
        out = (proc.stdout + proc.stderr).strip() or "(no output)"
        return _truncate(out)
    except subprocess.TimeoutExpired:
        return f"Error: timeout after {BASH_TIMEOUT_S}s"
    except Exception as e:
        return f"Error: {e}"


def tool_read(path: str, start_line: Optional[int] = None, end_line: Optional[int] = None) -> str:
    log_tool.info(f"[read] {path}")
    try:
        p = _safe_path(path)
        lines = p.read_text(encoding="utf-8", errors="replace").splitlines(keepends=True)
        i0 = max(0, (start_line or 1) - 1)
        i1 = end_line if end_line is not None else len(lines)
        numbered = "".join(f"{i0+1+i:5d}\t{ln}" for i, ln in enumerate(lines[i0:i1]))
        return _truncate(numbered or "(empty)")
    except FileNotFoundError:
        return f"Error: file not found: {path}"
    except Exception as e:
        return f"Error: {e}"


def tool_write(path: str, content: str) -> str:
    log_tool.info(f"[write] {path} ({len(content)} chars)")
    try:
        p = _safe_path(path)
        SNAPSHOTS[str(p)] = p.read_text(encoding="utf-8", errors="replace") if p.exists() else None
        action = "updated" if SNAPSHOTS[str(p)] is not None else "created"
        p.parent.mkdir(parents=True, exist_ok=True)
        p.write_text(content, encoding="utf-8")
        return f"{action}: {p} (snapshot saved -- use revert to undo)"
    except Exception as e:
        return f"Error: {e}"


def tool_grep(pattern: str, path: str = ".", recursive: bool = True) -> str:
    log_tool.info(f"[grep] {pattern!r}")
    try:
        p = _safe_path(path)
        flags = ["-rn"] if recursive else ["-n"]
        proc = subprocess.run(["grep", *flags, pattern, str(p)],
                              capture_output=True, text=True, timeout=30)
        return _truncate((proc.stdout + proc.stderr).strip() or "(no matches)", 10_000)
    except Exception as e:
        return f"Error: {e}"


def tool_glob(pattern: str) -> str:
    log_tool.info(f"[glob] {pattern}")
    full = str(WORKDIR / pattern) if not os.path.isabs(pattern) else pattern
    matches = sorted(_glob.glob(full, recursive=True))[:200]
    safe = [m for m in matches if Path(m).resolve().is_relative_to(WORKDIR)]
    return "\n".join(safe) if safe else "(no matches)"


def tool_revert(path: str) -> str:
    log_tool.info(f"[revert] {path}")
    try:
        p = _safe_path(path)
        key = str(p)
        if key not in SNAPSHOTS:
            return f"Error: no snapshot for {p}"
        prev = SNAPSHOTS.pop(key)
        if prev is None:
            p.unlink(missing_ok=True)
            return f"reverted: deleted {p} (was a new file)"
        p.write_text(prev, encoding="utf-8")
        return f"reverted: {p}"
    except Exception as e:
        return f"Error: {e}"

log.info("Core tools defined: bash, read, write, grep, glob, revert")


In [ ]:
"""
Tool schemas + base dispatch.

BASE = inspect BigQuery + build the catalog + file/shell. Subagents get exactly
this (so a subagent can profile a table AND write its findings into the shared
catalog). The LEAD registry adds spawn_subagent, planning and skills.
"""

def _fn(name, description, properties, required):
    return {"type": "function", "function": {
        "name": name, "description": description,
        "parameters": {"type": "object", "properties": properties, "required": required}}}


TOOLS_BASE: List[Dict[str, Any]] = [
    # ---- inspect (free metadata) ----
    _fn("bq_list_datasets", "List datasets in the project (respects the BQ_DATASETS allow-list). Free.",
        {}, []),
    _fn("bq_list_tables", "List table names + types in a dataset. Free.",
        {"dataset": {"type": "string"}, "max_tables": {"type": "integer", "description": "Default 300."}},
        ["dataset"]),
    _fn("bq_table_schema",
        "Full schema of one table: columns (name/type/mode/description, incl nested RECORDs), row "
        "count, size, partitioning, clustering, table description and labels. FREE metadata -- use "
        "this as your primary tool to understand a table.",
        {"table": {"type": "string", "description": "'dataset.table' or 'project.dataset.table'."}},
        ["table"]),
    _fn("bq_columns",
        "Find columns across datasets whose NAME matches a SQL LIKE pattern (e.g. '%_id', "
        "'customer%', 'account_key') -- the core join-key discovery tool. Cheap INFORMATION_SCHEMA "
        "metadata query.",
        {"name_like": {"type": "string", "description": "LIKE pattern: letters/digits/_/% only."},
         "dataset":   {"type": "string", "description": "One dataset, or '*'/omit for all (capped)."}},
        ["name_like"]),
    _fn("bq_estimate", "Dry-run a read-only SQL and report the bytes it WOULD scan, without running it.",
        {"sql": {"type": "string"}}, ["sql"]),
    # ---- read data (dry-run first, hard byte cap) ----
    _fn("bq_sample", "Return a few example rows from a table (SELECT * LIMIT n). Dry-run + byte-capped.",
        {"table": {"type": "string"}, "n": {"type": "integer", "description": "Rows, default 8, max 50."}},
        ["table"]),
    _fn("bq_query",
        "Run a read-only SELECT/WITH query. Dry-run first; REFUSED if it would scan more than the "
        "byte cap; runs with maximum_bytes_billed set. Use APPROX_COUNT_DISTINCT, COUNTIF, "
        "TABLESAMPLE, partition filters and LIMIT to stay cheap. Good for profiling (null %, "
        "distinct counts, value distributions) and verifying business rules.",
        {"sql": {"type": "string"}, "row_limit": {"type": "integer", "description": "Default 200."}},
        ["sql"]),
    _fn("bq_join_overlap",
        "Evidence for a join/FK: estimate how much two columns' value sets overlap (on a capped "
        "DISTINCT sample). High left->right coverage implies left is a FK into right. Dry-run + capped.",
        {"left_table": {"type": "string"}, "left_col": {"type": "string"},
         "right_table": {"type": "string"}, "right_col": {"type": "string"},
         "sample": {"type": "integer", "description": "Max distinct values per side (default 100000)."}},
        ["left_table", "left_col", "right_table", "right_col"]),
    # ---- build the catalog ----
    _fn("catalog_set_table",
        "Record/merge what you learned about a table: its purpose, grain (what one row represents), "
        "row_count, key_columns (PK/identifier columns), business_rules (observed constraints/"
        "semantics), and free-form notes.",
        {"table": {"type": "string"},
         "purpose": {"type": "string"},
         "grain": {"type": "string", "description": "What a single row represents."},
         "row_count": {"type": "integer"},
         "key_columns": {"type": "array", "items": {"type": "string"}},
         "business_rules": {"type": "array", "items": {"type": "string"}},
         "notes": {"type": "string"}},
        ["table"]),
    _fn("catalog_add_join",
        "Record a candidate join between two tables. confidence=high|medium|low; "
        "basis=name+type|fk-naming|pk-unique|value-overlap|manual. Cite evidence.",
        {"left": {"type": "string"}, "left_col": {"type": "string"},
         "right": {"type": "string"}, "right_col": {"type": "string"},
         "confidence": {"type": "string", "enum": list(JOIN_CONFIDENCE)},
         "basis": {"type": "string", "enum": list(JOIN_BASIS)},
         "evidence": {"type": "string"}},
        ["left", "left_col", "right", "right_col"]),
    _fn("catalog_summary", "Catalog stats: tables catalogued, business rules, join candidates by "
        "confidence, hub tables.", {}, []),
    _fn("catalog_export", "Write outputs to the sandbox. format = json | dictionary | joins | sql | all.",
        {"format": {"type": "string", "enum": ["json", "dictionary", "joins", "sql", "all"]}},
        []),
    # ---- file/shell ----
    _fn("bash", "Run a shell command in the sandbox WORKDIR (e.g. inspect the exported CSV).",
        {"command": {"type": "string"}}, ["command"]),
    _fn("read", "Read a sandbox file.",
        {"path": {"type": "string"}, "start_line": {"type": "integer"}, "end_line": {"type": "integer"}},
        ["path"]),
    _fn("write", "Write a sandbox file (notes/report). Snapshotted; revert to undo.",
        {"path": {"type": "string"}, "content": {"type": "string"}}, ["path", "content"]),
    _fn("grep", "Regex search across sandbox files.",
        {"pattern": {"type": "string"}, "path": {"type": "string"}, "recursive": {"type": "boolean"}},
        ["pattern"]),
    _fn("glob", "Find sandbox files matching a glob.", {"pattern": {"type": "string"}}, ["pattern"]),
    _fn("revert", "Restore a sandbox file to its pre-write state.", {"path": {"type": "string"}}, ["path"]),
]

DISPATCH_BASE: Dict[str, Callable[[Dict[str, Any]], str]] = {
    "bq_list_datasets": lambda a: tool_bq_list_datasets(),
    "bq_list_tables":   lambda a: tool_bq_list_tables(a["dataset"], a.get("max_tables", 300)),
    "bq_table_schema":  lambda a: tool_bq_table_schema(a["table"]),
    "bq_columns":       lambda a: tool_bq_columns(a["name_like"], a.get("dataset")),
    "bq_estimate":      lambda a: tool_bq_estimate(a["sql"]),
    "bq_sample":        lambda a: tool_bq_sample(a["table"], a.get("n", BQ_SAMPLE_ROWS)),
    "bq_query":         lambda a: tool_bq_query(a["sql"], a.get("row_limit", BQ_QUERY_ROW_LIMIT)),
    "bq_join_overlap":  lambda a: tool_bq_join_overlap(a["left_table"], a["left_col"],
                                                       a["right_table"], a["right_col"],
                                                       a.get("sample", 100_000)),
    "catalog_set_table": lambda a: run_catalog_set_table(
        a["table"], a.get("purpose"), a.get("grain"), a.get("row_count"),
        a.get("key_columns"), a.get("business_rules"), a.get("notes")),
    "catalog_add_join": lambda a: run_catalog_add_join(
        a["left"], a["left_col"], a["right"], a["right_col"],
        a.get("confidence", "medium"), a.get("basis", "name+type"), a.get("evidence", "")),
    "catalog_summary":  lambda a: run_catalog_summary(),
    "catalog_export":   lambda a: run_catalog_export(a.get("format", "all")),
    "bash":   lambda a: tool_bash(a["command"]),
    "read":   lambda a: tool_read(a["path"], a.get("start_line"), a.get("end_line")),
    "write":  lambda a: tool_write(a["path"], a["content"]),
    "grep":   lambda a: tool_grep(a["pattern"], a.get("path", "."), a.get("recursive", True)),
    "glob":   lambda a: tool_glob(a["pattern"]),
    "revert": lambda a: tool_revert(a["path"]),
}

log.info(f"Base tool registry: {list(DISPATCH_BASE)}")


In [ ]:
"""
Agent loop -- perception -> action -> observation -> repeat. Terminates when the
model returns a message with no tool_calls. MAX_TURNS caps runaway loops;
`messages` is mutated in place.
"""

def _run_one_tool_call(tc: Dict[str, Any], dispatch: Dict[str, Callable]) -> Dict[str, Any]:
    fn   = tc.get("function", {}) or {}
    name = fn.get("name", "")
    args = fn.get("arguments", {}) or {}
    if isinstance(args, str):
        try:
            args = json.loads(args)
        except json.JSONDecodeError:
            args = {}
    log_tool.info(f"-> {name}(" + ", ".join(f"{k}={str(v)[:40]!r}" for k, v in args.items()) + ")")
    if name not in dispatch:
        result = f"[error] Unknown tool: {name!r}. Available: {sorted(dispatch)}"
    else:
        try:
            result = dispatch[name](args)
        except Exception as e:
            result = f"[error] Tool {name!r} raised {type(e).__name__}: {e}"
            log_tool.exception(result)
    return {"role": "tool", "name": name, "content": _truncate(str(result), MAX_TOOL_OUTPUT)}


def agent_loop(messages, tools, dispatch, model=MODELS["lead"],
               max_turns=MAX_TURNS, label="lead") -> str:
    log_loop.info(f"[{label}] starting (model={model}, max_turns={max_turns}, tools={len(tools)})")
    final_text = ""
    for turn in range(1, max_turns + 1):
        log_loop.info(f"[{label}] turn {turn}/{max_turns}  msgs={len(messages)}")
        msg = ollama_chat(model=model, messages=messages, tools=tools)
        messages.append(msg)
        tool_calls = msg.get("tool_calls") or []
        text = (msg.get("content") or "").strip()
        if not tool_calls:
            log_loop.info(f"[{label}] DONE on turn {turn} -- no tool_calls.")
            return text
        log_loop.info(f"[{label}] turn {turn}: {len(tool_calls)} tool call(s)")
        for tc in tool_calls:
            messages.append(_run_one_tool_call(tc, dispatch))
    log_loop.warning(f"[{label}] hit MAX_TURNS={max_turns} without termination.")
    return final_text or "[agent_loop] max turns reached without a terminal response"

log.info("Agent loop ready.")


In [ ]:
"""
Subagents -- isolated context for deep single-table (or single-dataset) profiling.

When the lead wants to fully understand one table -- schema, sample, infer
purpose/grain/business rules, find its keys -- it calls spawn_subagent. The
subagent runs a fresh loop on the smaller model with the BASE tools (BigQuery +
catalog), so it records its findings directly into the shared catalog. Only its
final summary returns to the lead.
"""

SUBAGENT_SYSTEM = (
    "You are a focused data-analysis subagent profiling tables in a BigQuery warehouse for a "
    "financial institution. Tools: bq_table_schema (FREE -- start here), bq_sample, bq_query "
    "(read-only, byte-capped), bq_columns, bq_join_overlap, and catalog_set_table / "
    "catalog_add_join to record findings. You are given ONE table (or a small set). For each: "
    "infer its PURPOSE and GRAIN (what one row is) from column names/types/descriptions and a "
    "small sample; identify KEY columns (identifiers / likely primary keys); note BUSINESS RULES "
    "you can actually observe (status enums, monetary units/currencies, date grains, nullability, "
    "soft-delete flags, ranges); and propose JOIN candidates to other tables (use bq_columns to "
    "find columns with the same name elsewhere). Record everything with catalog_set_table / "
    "catalog_add_join, citing evidence. BE COST-AWARE: prefer free metadata; sample small; never "
    "scan a whole large table. Never invent a rule or a join -- only record what the data shows. "
    "When done, reply with a concise summary of what you catalogued. Do not ask questions."
)


def run_subagent(prompt: str) -> str:
    sub_id = uuid.uuid4().hex[:6]
    log_sub.info(f"[sub:{sub_id}] spawn -- {prompt[:120]!r}")
    sub_messages = [{"role": "system", "content": SUBAGENT_SYSTEM},
                    {"role": "user", "content": prompt}]
    try:
        result = agent_loop(sub_messages, TOOLS_BASE, DISPATCH_BASE,
                            model=MODELS["subagent"], label=f"sub:{sub_id}")
    except Exception as e:
        log_sub.exception(f"[sub:{sub_id}] crashed: {e}")
        return f"[subagent error] {type(e).__name__}: {e}"
    log_sub.info(f"[sub:{sub_id}] done -- {len(result)} chars")
    return result


TOOLS_LEAD: List[Dict[str, Any]] = TOOLS_BASE + [
    _fn("spawn_subagent",
        "Delegate exhaustive profiling of ONE table (or a small related set) to a fresh subagent "
        "(it has the same BigQuery + catalog tools and writes into the shared catalog). Use this to "
        "cover hundreds of tables without bloating your own context. Returns the subagent's summary.",
        {"prompt": {"type": "string",
                    "description": "Name the exact table(s) and instruct the subagent to infer "
                    "purpose/grain/rules/keys and record them, plus candidate joins."}},
        ["prompt"])]

DISPATCH_LEAD: Dict[str, Callable[[Dict[str, Any]], str]] = {
    **DISPATCH_BASE, "spawn_subagent": lambda a: run_subagent(a["prompt"])}

log.info(f"Subagent ready. Lead registry: {list(DISPATCH_LEAD)}")


In [ ]:
"""
Todo planning -- the working checklist for the CURRENT pass. Persisted to TODO_FILE.
"""

_TODO_STATUSES = ("pending", "in_progress", "done")

def _load_todos():
    if not TODO_FILE.exists():
        return []
    try:
        return json.loads(TODO_FILE.read_text(encoding="utf-8"))
    except (json.JSONDecodeError, OSError):
        return []

def _save_todos(items):
    TODO_FILE.write_text(json.dumps(items, indent=2), encoding="utf-8")

def run_todo_write(items):
    todos = [{"index": i, "text": t, "status": "pending"} for i, t in enumerate(items)]
    _save_todos(todos)
    return f"Wrote {len(todos)} todos:\n" + run_todo_read()

def run_todo_read():
    todos = _load_todos()
    if not todos:
        return "(no todos)"
    marks = {"pending": "[ ]", "in_progress": "[~]", "done": "[x]"}
    return "\n".join(f"{marks.get(t['status'], '[?]')} {t['index']}: {t['text']}" for t in todos)

def run_todo_update(index, status):
    if status not in _TODO_STATUSES:
        return f"Error: status must be one of {_TODO_STATUSES}, got {status!r}"
    todos = _load_todos()
    for t in todos:
        if t["index"] == index:
            t["status"] = status
            _save_todos(todos)
            return f"Todo #{index} -> {status}\n" + run_todo_read()
    return f"Error: no todo with index {index}"


TOOLS_LEAD += [
    _fn("todo_write",
        "Set the working todo list for this pass (e.g. one item per dataset/area). Replaces the list.",
        {"items": {"type": "array", "items": {"type": "string"}}}, ["items"]),
    _fn("todo_read", "Show the current todo list with statuses.", {}, []),
    _fn("todo_update", "Update one todo's status as you make progress.",
        {"index": {"type": "integer"}, "status": {"type": "string", "enum": list(_TODO_STATUSES)}},
        ["index", "status"]),
]
DISPATCH_LEAD.update({
    "todo_write":  lambda a: run_todo_write(a["items"]),
    "todo_read":   lambda a: run_todo_read(),
    "todo_update": lambda a: run_todo_update(a["index"], a["status"]),
})
log.info("Todo tools registered.")


In [ ]:
"""
Context compaction -- keeps a long, multi-hundred-table run within the context
budget. Keeps the last KEEP_RECENT messages verbatim, condenses older ones, and
persists the summary to MEMORY_FILE. The catalog itself lives on disk regardless.
"""

def _estimate_size(messages):
    return sum(len(str(m.get("content", "") or "")) for m in messages)

def _render_for_summary(messages):
    parts = []
    for m in messages:
        role = m.get("role", "?")
        content = str(m.get("content", "") or "")
        if m.get("tool_calls"):
            names = [tc.get("function", {}).get("name", "?") for tc in m["tool_calls"]]
            content = (content + f"  (called tools: {', '.join(names)})").strip()
        parts.append(f"[{role}] {content}")
    return "\n\n".join(parts)

def _summarize(messages):
    transcript = _render_for_summary(messages)
    log_compact.info(f"summarising {len(messages)} msgs (~{len(transcript)} chars)")
    summary_messages = [
        {"role": "system", "content": (
            "You are a context compressor for a BigQuery-cataloguing agent. Condense the conversation "
            "into a concise summary. PRESERVE: which datasets/tables have been catalogued, key facts "
            "found (purpose/grain/keys/business rules), join candidates and their confidence, and what "
            "remains to cover. DROP redundant tool chatter. Write prose.")},
        {"role": "user", "content": f"Summarise this history:\n\n{transcript[:20000]}"},
    ]
    msg = ollama_chat(model=MODELS["summarizer"], messages=summary_messages, tools=None)
    return (msg.get("content") or "").strip()

def maybe_compress(messages):
    if _estimate_size(messages) < COMPRESS_THRESHOLD or len(messages) <= KEEP_RECENT + 1:
        return False
    head, body = [], messages
    if messages and messages[0].get("role") == "system":
        head, body = [messages[0]], messages[1:]
    old, recent = body[:-KEEP_RECENT], body[-KEEP_RECENT:]
    if not old:
        return False
    summary = _summarize(old)
    try:
        MEMORY_FILE.write_text(f"# Agent context memory\n\n{summary}\n", encoding="utf-8")
    except OSError:
        pass
    messages.clear()
    messages.extend(head)
    messages.append({"role": "user", "content": f"[Summary of earlier turns]:\n\n{summary}"})
    messages.append({"role": "assistant", "content": "Understood. Continuing the cataloguing from here."})
    messages.extend(recent)
    log_compact.info(f"compressed {len(old)} msgs; history now {len(messages)}")
    return True

log.info("Context compaction ready.")


In [ ]:
"""
Task graph -- a persistent, dependency-aware backlog for multi-session coverage of
hundreds of tables (e.g. one task per dataset; task_next yields the next area).
Survives restarts via TASKS_FILE.
"""

_TASK_STATUSES = ("pending", "in_progress", "done", "failed")

def _load_tasks():
    if not TASKS_FILE.exists():
        return []
    try:
        return json.loads(TASKS_FILE.read_text(encoding="utf-8"))
    except (json.JSONDecodeError, OSError):
        return []

def _save_tasks(tasks):
    TASKS_FILE.write_text(json.dumps(tasks, indent=2), encoding="utf-8")

def run_task_create(description, depends_on=None, priority="medium"):
    tasks = _load_tasks()
    tid = uuid.uuid4().hex[:8]
    tasks.append({"id": tid, "description": description, "status": "pending",
                  "priority": priority, "depends_on": depends_on or [], "result": ""})
    _save_tasks(tasks)
    return f"Created task {tid}: {description}"

def run_task_list():
    tasks = _load_tasks()
    if not tasks:
        return "(no tasks)"
    rows = []
    for t in tasks:
        deps = f" needs={','.join(t['depends_on'])}" if t.get("depends_on") else ""
        rows.append(f"[{t['id']}] {t['status']:11s} {t['priority']:6s}{deps}  {t['description']}")
    return "\n".join(rows)

def run_task_update(task_id, status, result=""):
    if status not in _TASK_STATUSES:
        return f"Error: status must be one of {_TASK_STATUSES}, got {status!r}"
    tasks = _load_tasks()
    for t in tasks:
        if t["id"].startswith(task_id):
            t["status"] = status
            if result:
                t["result"] = result
            _save_tasks(tasks)
            return f"Task {t['id']} -> {status}"
    return f"Error: no task matching id {task_id!r}"

def run_task_next():
    tasks = _load_tasks()
    done = {t["id"] for t in tasks if t["status"] == "done"}
    for t in tasks:
        if t["status"] == "pending" and all(d in done for d in t.get("depends_on", [])):
            return f"Next: [{t['id']}] ({t['priority']}) {t['description']}"
    return "No unblocked tasks."


TOOLS_LEAD += [
    _fn("task_create", "Create a durable task (for multi-session coverage; heavier than todo).",
        {"description": {"type": "string"},
         "depends_on": {"type": "array", "items": {"type": "string"}},
         "priority": {"type": "string", "enum": ["high", "medium", "low"]}}, ["description"]),
    _fn("task_list", "List all durable tasks.", {}, []),
    _fn("task_update", "Update a task's status (and optional result).",
        {"task_id": {"type": "string"}, "status": {"type": "string", "enum": list(_TASK_STATUSES)},
         "result": {"type": "string"}}, ["task_id", "status"]),
    _fn("task_next", "Next unblocked task (deps satisfied).", {}, []),
]
DISPATCH_LEAD.update({
    "task_create": lambda a: run_task_create(a["description"], a.get("depends_on"), a.get("priority", "medium")),
    "task_list":   lambda a: run_task_list(),
    "task_update": lambda a: run_task_update(a["task_id"], a["status"], a.get("result", "")),
    "task_next":   lambda a: run_task_next(),
})
log.info("Task graph registered.")


In [ ]:
"""
Skill loading -- optional, lazy domain playbooks under SKILLS_DIR (e.g. a
"banking-core-schema" or "PII-handling" checklist). The system prompt gets a
cheap index; load_skill pulls a full guide on demand.
"""

def _skill_dir(name):
    if not name or "/" in name or ".." in name:
        return None
    d = (SKILLS_DIR / name).resolve()
    try:
        d.relative_to(SKILLS_DIR.resolve())
    except ValueError:
        return None
    return d if (d / "SKILL.md").is_file() else None

def run_list_skills():
    if not SKILLS_DIR.is_dir():
        return "(no skills available)"
    entries = []
    for child in sorted(SKILLS_DIR.iterdir()):
        md = child / "SKILL.md"
        if not md.is_file():
            continue
        summary = ""
        for line in md.read_text(encoding="utf-8", errors="replace").splitlines():
            if line.strip():
                summary = line.lstrip("# ").strip()
                break
        entries.append(f"- {child.name}: {summary}")
    return "\n".join(entries) if entries else "(no skills available)"

def run_load_skill(name):
    d = _skill_dir(name)
    if d is None:
        return f"Error: no skill named {name!r}. Available:\n{run_list_skills()}"
    return _truncate((d / "SKILL.md").read_text(encoding="utf-8", errors="replace"))

def skills_index_for_prompt():
    idx = run_list_skills()
    if idx == "(no skills available)":
        return ""
    return "\n\nAvailable skills (load_skill(name) to read the full guide):\n" + idx


TOOLS_LEAD += [
    _fn("list_skills", "List available skill guides.", {}, []),
    _fn("load_skill", "Load one skill guide's full text by name.", {"name": {"type": "string"}}, ["name"]),
]
DISPATCH_LEAD.update({
    "list_skills": lambda a: run_list_skills(),
    "load_skill":  lambda a: run_load_skill(a["name"]),
})
log.info("Skill loading registered.")


In [ ]:
"""
chat() -- the user-facing entry point. HISTORY persists across calls so a long
cataloguing run is one continuous conversation. reset_chat() wipes the
conversation+memory; new_catalog() starts a fresh catalog. The catalog lives in
CATALOG_FILE and persists regardless.
"""

LEAD_SYSTEM = (
    "You are a senior data analyst / data modeler making sense of a large BigQuery warehouse "
    "(hundreds of tables) belonging to a financial institution. Goal: produce a trustworthy "
    "catalog that (1) explains each table's PURPOSE and GRAIN, (2) captures the BUSINESS RULES "
    "and semantics of the data, and (3) identifies the PRINCIPAL JOIN ATTRIBUTES that link tables.\n\n"
    "TOOLS:\n"
    "- Inspect (FREE metadata -- do most of your work here): bq_list_datasets, bq_list_tables, "
    "bq_table_schema (primary tool), bq_columns (join-key discovery by column name), bq_estimate.\n"
    "- Read data (byte-capped, dry-run first -- use sparingly): bq_sample (a few rows), bq_query "
    "(profiling: COUNTIF, APPROX_COUNT_DISTINCT, value distributions, rule checks), bq_join_overlap "
    "(value-overlap evidence for a join).\n"
    "- Build the catalog: catalog_set_table (purpose/grain/keys/business_rules/notes), "
    "catalog_add_join (candidate joins w/ confidence+basis+evidence), catalog_summary, catalog_export.\n"
    "- Scale & plan: todo_write/update (this pass), task_create/next (durable coverage across all "
    "datasets), spawn_subagent (delegate one table's deep profiling -- it writes into the same "
    "catalog), plus file/shell and skills.\n\n"
    "METHOD:\n"
    "1. bq_list_datasets, then bq_list_tables per dataset to size the job. Plan coverage with "
    "todo/tasks; with hundreds of tables, delegate per-table profiling to subagents and proceed "
    "systematically -- do not stop after a handful.\n"
    "2. For each table: bq_table_schema FIRST (free). Infer purpose and grain (what one row is) from "
    "names/types/descriptions; identify key/identifier columns. Take a small bq_sample to ground your "
    "reading and spot business rules (status enums, currencies/units, date grains, nullability, "
    "soft-deletes, ranges). Record with catalog_set_table.\n"
    "3. Joins: for each identifier column (e.g. *_id, *_key, account_no), use bq_columns to find the "
    "same/compatible column in other tables. Record candidates with catalog_add_join: basis=fk-naming "
    "or name+type, confidence per how strong the signal is. For a FEW high-value/uncertain links, "
    "confirm with bq_join_overlap and upgrade to basis=value-overlap with the measured coverage as "
    "evidence. Be cost-aware -- overlap checks scan data.\n"
    "4. Never invent a purpose, rule, or join -- only record what schema/sample/data supports; mark "
    "uncertainty in notes.\n"
    "5. Periodically catalog_summary. When coverage (or the requested scope) is complete, call "
    "catalog_export('all') to write data_catalog.json, data_dictionary.md, join_candidates.csv and "
    "suggested_views.sql, then give the user a clear written summary: the main subject areas, the hub "
    "tables, the principal join keys, and notable business rules.\n"
    "6. When done, STOP calling tools and reply with that summary."
)

HISTORY: List[Dict[str, Any]] = []

def _ensure_history():
    if not HISTORY:
        system = LEAD_SYSTEM + skills_index_for_prompt()
        HISTORY.append({"role": "system", "content": system})
        log_chat.info(f"history seeded (system prompt {len(system)} chars)")

def reset_chat():
    """Wipe the conversation + memory. Does NOT delete the catalog; use new_catalog() for that."""
    HISTORY.clear()
    try:
        MEMORY_FILE.unlink(missing_ok=True)
    except OSError:
        pass
    log_chat.info("chat history reset")

def new_catalog():
    """Start a fresh catalog (deletes CATALOG_FILE)."""
    try:
        CATALOG_FILE.unlink(missing_ok=True)
    except OSError:
        pass
    log_cat.info("catalog reset")

def chat(query: str) -> str:
    _ensure_history()
    log_chat.info(f"=== USER: {query[:200]!r} ===")
    HISTORY.append({"role": "user", "content": query})
    answer = agent_loop(HISTORY, TOOLS_LEAD, DISPATCH_LEAD, model=MODELS["lead"], label="lead")
    if maybe_compress(HISTORY):
        log_chat.info("history compacted after this turn")
    print("\n" + "=" * 70 + "\nASSISTANT:\n" + "=" * 70 + "\n" + answer)
    return answer

log.info("chat() ready. chat('...') to run; reset_chat() new conversation; new_catalog() fresh catalog.")


In [ ]:
"""
Driver -- connectivity checks, then kick off a cataloguing run (guarded).

Order: ollama healthcheck -> BigQuery client + list datasets. The agent run only
starts if BigQuery is reachable. Otherwise it prints exactly what to fix and skips
(no hard failure).
"""
assert ollama_healthcheck(), "Ollama not reachable / models missing -- fix the config cell first."

print("\n-- BigQuery connectivity --")
if not _BQ_AVAILABLE:
    print(f"google-cloud-bigquery NOT installed: {_BQ_IMPORT_ERR}")
    print(">>> pip install google-cloud-bigquery, then re-run.")
    ds_info = "Error: client unavailable"
else:
    ds_info = tool_bq_list_datasets()
    print(ds_info[:1500])

ready = _BQ_AVAILABLE and not ds_info.startswith("Error")
if not ready:
    print("\n>>> BigQuery not ready. In the config cell set BQ_PROJECT (and optionally BQ_DATASETS / "
          "BQ_LOCATION), and authenticate via `gcloud auth application-default login` or "
          "GOOGLE_APPLICATION_CREDENTIALS=<service-account.json>. Then re-run this cell.")
    print(">>> Skipping the agent run for now.")
else:
    new_catalog()     # fresh catalog for this demo run
    reset_chat()
    answer = chat(
        "Begin cataloguing this BigQuery warehouse. List the datasets and tables to size the job and "
        "plan coverage with todo/tasks. Then work through the tables (delegating per-table profiling "
        "to subagents): for each, infer its purpose and grain, capture observable business rules, and "
        "record key columns -- using mostly FREE schema metadata and only small, byte-capped samples. "
        "Discover the principal join attributes with bq_columns and record join candidates with "
        "confidence; confirm a few important ones with bq_join_overlap. When done (or when you've "
        "covered the requested scope), call catalog_export('all') and summarise the subject areas, hub "
        "tables, principal join keys and notable business rules."
    )
    print("\n" + "-" * 70)
    print("Artifacts in:", WORKDIR)
    for fn in ("data_catalog.json", "data_dictionary.md", "join_candidates.csv", "suggested_views.sql"):
        p = WORKDIR / fn
        print(f"  {'OK ' if p.exists() else '-- '} {fn}" + (f"  ({p.stat().st_size} B)" if p.exists() else ""))
    print(f"history: {len(HISTORY)} msgs (~{_estimate_size(HISTORY)} chars)")
